# Transformer optimizer benchmark

Run this notebook only on a **free Colab T4 GPU**. It clones the implementation branch, runs every experiment in fp32, validates all artifacts, captures this executed notebook, and can push the evidence back without printing or storing the GitHub token. Expected runtime: 45–90 minutes.

## 1. Configuration and checkout
Set the repository URL below. Add a Colab Secret named `GITHUB_TOKEN` only if the final cell should push results. Select **Runtime → Change runtime type → T4 GPU** before continuing.

In [1]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/LokeshJatangi/transformer_optimizer_benchmark.git"
BRANCH = "colab-results"
PUSH_RESULTS = True
WORKDIR = Path("/content/transformer-optimizer-benchmark")
assert REPO_URL.startswith("https://github.com/"), "Set REPO_URL first"
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
source_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
os.environ["SOURCE_COMMIT"] = source_commit
print("Checked out source commit:", source_commit)


Checked out source commit: ac320c4c9d9b181554746c99301ccdcbf86aa7d5


## 2. Environment checks and local unit suite

In [2]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime"
device_name = torch.cuda.get_device_name(0)
assert "T4" in device_name.upper(), f"Expected a free Colab T4, got {device_name}"
print("Validated device:", device_name)
subprocess.run(["python", "-m", "pytest", "-q"], check=True)


Validated device: Tesla T4


CompletedProcess(args=['python', '-m', 'pytest', '-q'], returncode=0)

## 3. Full synchronized T4 experiment
This is the only cell that may create benchmark results. Candidate comparisons use identical initialization, batch order, validation data, and tuning budgets.

In [3]:
from transformer_optimizer_benchmark import run_full
metrics = run_full()
print("Completed in", metrics["timing"]["total_readable"])
print("Retained scheduler:", metrics["scheduler_comparison"]["winner"])
print("Width-4096 LR prediction:", metrics["width_sweep"]["fit"]["predicted_lr_width_4096"])
print("Confidence:", metrics["width_sweep"]["fit"]["confidence"])


Completed in 5m 53.3s
Retained scheduler: cosine
Width-4096 LR prediction: 2.5000000000000072e-05
Confidence: moderate


## 4. Artifact and assertion audit

In [4]:
import json
from transformer_optimizer_benchmark import validate_metrics
saved = json.loads(Path("results/metrics.json").read_text())
validate_metrics(saved)
required = ["metrics.json", "run_log.json", "run_log.csv", "adam_bias_correction.png",
            "relative_updates_cosine.png", "relative_updates_wsd.png",
            "width_lr_sweep.png", "retained_model.pt"]
missing = [name for name in required if not (Path("results") / name).exists()]
assert not missing, missing
assert Path("README.md").read_text().startswith("# Transformer optimizer benchmark")
failed = [r for r in saved["runs"] if r["status"] != "ok"]
assert not failed, failed
print(f"All internal assertions passed; {len(saved['runs'])} timing records and {len(required)} required artifacts verified.")
print("Review results/run_log.csv and the four PNG plots before accepting the commit.")


All internal assertions passed; 75 timing records and 8 required artifacts verified.
Review results/run_log.csv and the four PNG plots before accepting the commit.


## 5. Capture and optionally push evidence
The token is obtained from Colab Secrets, held only in the child process environment, and never printed, written to disk, put in the remote URL, or added to Git configuration.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from google.colab import _message, userdata

# Undo staging from the failed packaging attempt without deleting any files.
subprocess.run(["git", "restore", "--staged", ":/"], check=True)

# README will be written later after reviewing the evidence.
# Discard the automatically generated version.
subprocess.run(["git", "restore", "README.md"], check=True)

# Capture the actually executed notebook, including outputs.
snapshot = _message.blocking_request("get_ipynb", timeout_sec=120)
notebook = snapshot.get("ipynb", snapshot)
assert isinstance(notebook, dict) and "cells" in notebook

notebook_path = Path("transformer_optimizer_benchmark_colab.ipynb")
notebook_path.write_text(
    json.dumps(notebook, ensure_ascii=False, indent=1),
    encoding="utf-8",
)

# Stage evidence only. README.md is deliberately excluded.
subprocess.run(
    ["git", "add", "transformer_optimizer_benchmark_colab.ipynb", "results"],
    check=True,
)

# Check for whitespace problems and normalize only text artifacts if necessary.
check = subprocess.run(
    ["git", "diff", "--cached", "--check"],
    text=True,
    capture_output=True,
)

if check.returncode != 0:
    print("Initial whitespace check reported:")
    print(check.stdout or check.stderr)

    text_suffixes = {".ipynb", ".json", ".csv", ".md", ".txt"}
    text_files = [notebook_path]
    text_files.extend(
        path
        for path in Path("results").rglob("*")
        if path.is_file() and path.suffix.lower() in text_suffixes
    )

    for path in text_files:
        content = path.read_text(encoding="utf-8")
        had_final_newline = content.endswith(("\n", "\r"))
        normalized = "\n".join(
            line.rstrip(" \t") for line in content.splitlines()
        )
        if had_final_newline:
            normalized += "\n"
        path.write_text(normalized, encoding="utf-8")

    subprocess.run(
        ["git", "add", "transformer_optimizer_benchmark_colab.ipynb", "results"],
        check=True,
    )

final_check = subprocess.run(
    ["git", "diff", "--cached", "--check"],
    text=True,
    capture_output=True,
)

assert final_check.returncode == 0, (
    "Whitespace validation still failed:\n"
    + final_check.stdout
    + final_check.stderr
)

subprocess.run(
    ["git", "config", "user.name", "Colab Results Bot"],
    check=True,
)
subprocess.run(
    ["git", "config", "user.email", "colab-results@users.noreply.github.com"],
    check=True,
)

subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "results: add verified Colab T4 optimizer evidence",
    ],
    check=True,
)

result_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

if PUSH_RESULTS:
    token = userdata.get("GITHUB_TOKEN")
    assert token, (
        "Add GITHUB_TOKEN to Colab Secrets or set PUSH_RESULTS=False"
    )

    env = dict(os.environ, GITHUB_TOKEN=token)
    helper = (
        "!f() { "
        "echo username=x-access-token; "
        "echo password=$GITHUB_TOKEN; "
        "}; f"
    )

    subprocess.run(
        [
            "git",
            "-c",
            f"credential.helper={helper}",
            "push",
            "origin",
            f"HEAD:{BRANCH}",
        ],
        check=True,
        env=env,
    )

    del token, env

print("Verified results commit:", result_commit)
print("README.md was not generated or staged by Colab.")

In [5]:
from google.colab import _message, userdata
snapshot = _message.blocking_request("get_ipynb", timeout_sec=120)
notebook = snapshot.get("ipynb", snapshot)
assert isinstance(notebook, dict) and "cells" in notebook
Path("transformer_optimizer_benchmark_colab.ipynb").write_text(json.dumps(notebook, ensure_ascii=False, indent=1))
subprocess.run(["git", "config", "user.name", "Colab Results Bot"], check=True)
subprocess.run(["git", "config", "user.email", "colab-results@users.noreply.github.com"], check=True)
subprocess.run(["git", "add", "README.md", "transformer_optimizer_benchmark_colab.ipynb", "results"], check=True)
subprocess.run(["git", "diff", "--cached", "--check"], check=True)
subprocess.run(["git", "commit", "-m", "results: add verified Colab T4 optimizer evidence"], check=True)
result_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if PUSH_RESULTS:
    token = userdata.get("GITHUB_TOKEN")
    assert token, "Add GITHUB_TOKEN to Colab Secrets or set PUSH_RESULTS=False"
    env = dict(os.environ, GITHUB_TOKEN=token)
    helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
    subprocess.run(["git", "-c", f"credential.helper={helper}", "push", "origin", f"HEAD:{BRANCH}"], check=True, env=env)
    del token, env
print("Verified results commit:", result_commit)


CalledProcessError: Command '['git', 'diff', '--cached', '--check']' returned non-zero exit status 2.